In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

resnet = models.resnet18(pretrained=True)
print(resnet)
print(f"\nTotal parameters: {sum(p.numel() for p in resnet.parameters()):,}")

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:00<00:00, 161MB/s] 


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
resnet.fc = nn.Linear(512, 10) 


for param in resnet.parameters():
    param.requires_grad = False

for param in resnet.fc.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet.parameters())
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters   : {total - trainable:,}")
print(f"Total parameters    : {total:,}")

model = resnet.to(device)

Trainable parameters: 5,130
Frozen parameters   : 11,176,512
Total parameters    : 11,181,642


In [ ]:
resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
resnet.maxpool = nn.Identity()  
print("Modified for 32×32 input")
print(f"conv1: {resnet.conv1}")
print(f"maxpool: {resnet.maxpool}")

Modified for 32×32 input
conv1: Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
maxpool: Identity()


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   
                         std=[0.229, 0.224, 0.225])     
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=128, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches : {len(test_loader)}")

100%|██████████| 170M/170M [00:19<00:00, 8.93MB/s] 


Train batches: 391
Test batches : 79


In [6]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        loss    = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()
        total_loss += loss.item()
    
    return total_loss / len(loader), 100 * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct    = 0
    total      = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs  = model(images)
            loss     = criterion(outputs, labels)
            
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()
            total_loss += loss.item()
    
    return total_loss / len(loader), 100 * correct / total

In [ ]:
resnet = models.resnet18(weights='IMAGENET1K_V1')
resnet.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
resnet.maxpool = nn.Identity()
resnet.fc      = nn.Linear(512, 10)

for param in resnet.parameters():
    param.requires_grad = True

model = resnet.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,}  ← all parameters")


optimizer = torch.optim.Adam([
    {'params': resnet.conv1.parameters(), 'lr': 0.001},  
    {'params': resnet.layer1.parameters(), 'lr': 0.0001},  
    {'params': resnet.layer2.parameters(), 'lr': 0.0001},
    {'params': resnet.layer3.parameters(), 'lr': 0.0001},
    {'params': resnet.layer4.parameters(), 'lr': 0.0001},
    {'params': resnet.fc.parameters(), 'lr': 0.001},  
])

criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5)

print("Approach B — Fine-tuning all layers")
print("New layers (conv1, fc): lr=0.001")
print("Pretrained layers      : lr=0.0001")

Trainable: 11,173,962  ← all parameters
Approach B — Fine-tuning all layers
New layers (conv1, fc): lr=0.001
Pretrained layers      : lr=0.0001


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5)

n_epochs      = 20  
best_val_acc  = 0.0
patience      = 5
patience_counter = 0

for epoch in range(n_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc     = evaluate(model, test_loader, criterion, device)
    
    scheduler.step(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_resnet_A.pth')
    else:
        patience_counter += 1
    
    print(f"Epoch {epoch+1:2d}/{n_epochs} | "
          f"Train: {train_acc:.1f}% | "
          f"Val: {val_acc:.1f}% | "
          f"Patience: {patience_counter}/{patience}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nApproach A (Feature Extraction): {best_val_acc:.2f}%")

Epoch  1/20 | Train: 72.0% | Val: 78.4% | Patience: 0/5
Epoch  2/20 | Train: 83.8% | Val: 81.2% | Patience: 0/5
Epoch  3/20 | Train: 86.9% | Val: 85.8% | Patience: 0/5
Epoch  4/20 | Train: 89.3% | Val: 86.3% | Patience: 0/5
Epoch  5/20 | Train: 90.4% | Val: 87.1% | Patience: 0/5
Epoch  6/20 | Train: 91.3% | Val: 89.5% | Patience: 0/5
Epoch  7/20 | Train: 92.2% | Val: 89.7% | Patience: 0/5
Epoch  8/20 | Train: 93.0% | Val: 90.0% | Patience: 0/5
Epoch  9/20 | Train: 93.4% | Val: 90.8% | Patience: 0/5
Epoch 10/20 | Train: 93.9% | Val: 90.5% | Patience: 1/5
Epoch 11/20 | Train: 94.3% | Val: 90.6% | Patience: 2/5
Epoch 12/20 | Train: 94.9% | Val: 90.8% | Patience: 0/5
Epoch 13/20 | Train: 95.2% | Val: 90.2% | Patience: 1/5
Epoch 14/20 | Train: 95.6% | Val: 90.9% | Patience: 0/5
Epoch 15/20 | Train: 95.7% | Val: 91.2% | Patience: 0/5
Epoch 16/20 | Train: 96.1% | Val: 91.3% | Patience: 0/5
Epoch 17/20 | Train: 96.3% | Val: 91.2% | Patience: 1/5
Epoch 18/20 | Train: 96.4% | Val: 91.9% | Patien